# 🤖 Game Helper Telegram MAX

Telegram-версия Game Helper MAX.

🔐 Токен вводится скрыто через `getpass` и не записывается в код.

In [ ]:
# ============================================================
# 🎮 GAME HELPER MAX
# One-cell edition
# Python version first — Telegram later
# ============================================================

import os
import json
import random
import joblib

try:
    from sklearn.tree import DecisionTreeClassifier
    SKLEARN_OK = True
except ImportError:
    SKLEARN_OK = False
    DecisionTreeClassifier = None

# -----------------------------
# 📁 FILES
# -----------------------------
PROGRESS_FILE = "progress.json"
ML_MODEL_FILE = "game_helper_ml.joblib"
ML_DATA_FILE = "ml_training_data.json"

# -----------------------------
# 🎮 GAMES
# -----------------------------
games = {
    1: {
        "name": "Poppy Playtime",
        "description": "Хоррор-игра с исследованием фабрики Playtime Co."
    },
    2: {
        "name": "Minecraft",
        "description": "Исследование мира, строительство и выживание."
    },
    3: {
        "name": "Roblox",
        "description": "Платформа с огромным количеством пользовательских игр."
    }
}

# -----------------------------
# 📚 GUIDES — 30 TOTAL
# -----------------------------
guides = {
    1: {
        "name": "Poppy Playtime",
        "guides": {
            1: "Chapter 1 — A Tight Squeeze",
            2: "Chapter 2 — Fly in a Web",
            3: "Chapter 3 — Deep Sleep",
            4: "Персонажи и монстры",
            5: "Головоломки",
            6: "Секреты и Easter Eggs",
            7: "Лор Playtime Co.",
            8: "Советы новичку"
        }
    },
    2: {
        "name": "Minecraft",
        "guides": {
            1: "Первые 10 минут",
            2: "Первая ночь",
            3: "Добыча руд",
            4: "Алмазы",
            5: "Оружие и броня",
            6: "Nether",
            7: "The End",
            8: "Строительство",
            9: "Зачарования",
            10: "Фермы",
            11: "Мобы",
            12: "Деревни",
            13: "Редстоун",
            14: "Выживание"
        }
    },
    3: {
        "name": "Roblox",
        "guides": {
            1: "Основы Roblox",
            2: "Как найти хорошую игру",
            3: "Roblox Studio",
            4: "Безопасность",
            5: "Жанры Roblox",
            6: "Создание собственной игры",
            7: "Советы новичку",
            8: "FAQ"
        }
    }
}

# -----------------------------
# 🧠 QUIZ DATA — 90 TOTAL
# 30 PER GAME:
# 8 Easy + 8 Normal + 7 Hard + 7 Expert
# -----------------------------
raw_quizzes = {
    1: [
        ("Easy", "Что такое Playtime Co.?", "Компания", "Школа", "Город", "Игра"),
        ("Easy", "Как называется главный инструмент игрока в Chapter 1?", "GrabPack", "Hammer", "Sword", "Bow"),
        ("Easy", "Кто является известным синим персонажем?", "Huggy Wuggy", "Steve", "Piggy", "Noob"),
        ("Easy", "Где происходит действие первой главы?", "На фабрике", "В деревне", "В космосе", "В замке"),
        ("Easy", "Что используется для активации некоторых механизмов?", "GrabPack", "Кирка", "Лук", "Щит"),
        ("Easy", "Какой жанр у Poppy Playtime?", "Horror", "Racing", "Sports", "Strategy"),
        ("Easy", "Как называется кукла из названия игры?", "Poppy", "Bonnie", "Alex", "Steve"),
        ("Easy", "Что обычно нужно делать в игре?", "Исследовать и решать головоломки", "Только бегать", "Только строить", "Только стрелять"),

        ("Normal", "Что делает GrabPack?", "Позволяет взаимодействовать с объектами на расстоянии", "Создаёт мобов", "Телепортирует", "Строит дома"),
        ("Normal", "Что особенно важно при прохождении фабрики?", "Исследование и головоломки", "Рыбалка", "Фермы", "Гонки"),
        ("Normal", "Кто связан с Playtime Co.?", "Игрушки", "Космонавты", "Пираты", "Футболисты"),
        ("Normal", "Что можно найти на фабрике?", "Игрушки и записи", "Только еду", "Только оружие", "Только транспорт"),
        ("Normal", "Для чего нужны цветные руки GrabPack?", "Для разных взаимодействий", "Для украшения", "Для еды", "Для бега"),
        ("Normal", "Что помогает продвигаться по локации?", "Решение головоломок", "Покупка машины", "Крафт брони", "Посадка деревьев"),
        ("Normal", "Что характерно для игры?", "Напряжение и исследование", "Спортивные матчи", "Гонки", "Строительство города"),
        ("Normal", "Что игрок обычно ищет во время исследования?", "Способ открыть следующий участок", "Футбольный мяч", "Алмазы", "Лошадь"),

        ("Hard", "Почему игроку важно внимательно изучать окружение?", "Подсказки могут быть частью головоломок", "Чтобы быстрее копать", "Чтобы найти деревню", "Чтобы получить XP"),
        ("Hard", "Что является частью атмосферы Poppy Playtime?", "Заброшенная фабрика игрушек", "Средневековый замок", "Океан", "Космостанция"),
        ("Hard", "Какую роль часто играют записи и документы?", "Раскрывают детали истории", "Дают броню", "Меняют погоду", "Телепортируют"),
        ("Hard", "Что отличает головоломки от простого исследования?", "Нужно понять механику или последовательность действий", "Нужно просто идти вперёд", "Нужно добыть дерево", "Нужно победить босса"),
        ("Hard", "Почему GrabPack важен?", "Он является ключевой частью взаимодействия с окружением", "Он только косметический", "Он нужен только в меню", "Он заменяет карту"),
        ("Hard", "Что лучше делать при непонятной головоломке?", "Проверять окружение и доступные механизмы", "Сразу закрывать игру", "Прыгать без цели", "Искать Minecraft"),
        ("Hard", "Что помогает понять лор?", "Документы, записи и детали окружения", "Только меню", "Только настройки", "Только звук"),

        ("Expert", "Какой подход лучше всего подходит для сложной головоломки?", "Разделить её на механизмы и проверить связи между ними", "Нажимать всё подряд", "Игнорировать окружение", "Перезапускать ПК"),
        ("Expert", "Почему детали фабрики важны для истории?", "Они могут связывать игровой процесс с событиями прошлого", "Они только украшают карту", "Они нужны для FPS", "Они заменяют сюжет"),
        ("Expert", "Что полезнее всего при изучении лора?", "Сопоставлять документы, окружение и события", "Читать только название главы", "Смотреть только меню", "Не обращать внимания на записи"),
        ("Expert", "Что делает хоррор-головоломку эффективной?", "Необходимость думать под давлением атмосферы", "Только высокая скорость", "Только количество врагов", "Только графика"),
        ("Expert", "Почему один и тот же объект может быть важен несколько раз?", "Он может быть частью нескольких связанных механизмов", "Потому что игра случайно повторяет объекты", "Потому что карта маленькая", "Потому что он декоративный"),
        ("Expert", "Что лучше всего описывает исследование Poppy Playtime?", "Поиск связей между окружением, механиками и историей", "Только поиск предметов", "Только бег", "Только бой"),
        ("Expert", "Какой навык особенно полезен для сложных частей игры?", "Наблюдательность и логическое мышление", "Только скорость реакции", "Только строительство", "Только добыча ресурсов"),
    ],

    2: [
        ("Easy", "Что нужно сделать в первую ночь?", "Построить укрытие", "Найти Ender Dragon", "Создать маяк", "Построить город"),
        ("Easy", "Какая руда нужна для железных инструментов?", "Железо", "Уголь", "Редстоун", "Лазурит"),
        ("Easy", "Что используется для добычи дерева?", "Топор", "Лук", "Щит", "Ведро"),
        ("Easy", "Как называется измерение с лавой и крепостями?", "Nether", "The End", "Overworld", "Moon"),
        ("Easy", "Что нужно для верстака?", "Дерево", "Алмазы", "Обсидиан", "Редстоун"),
        ("Easy", "Что восстанавливает голод?", "Еда", "Камень", "Факел", "Стекло"),
        ("Easy", "Как называется финальный дракон?", "Ender Dragon", "Warden", "Wither", "Creeper"),
        ("Easy", "Что освещает тёмные места?", "Факел", "Кирка", "Кровать", "Сундук"),

        ("Normal", "Зачем нужна кровать?", "Чтобы спать и установить точку возрождения", "Для добычи руды", "Для крафта меча", "Для полёта"),
        ("Normal", "Что обычно добывают глубоко под землёй?", "Руды", "Облака", "Снег", "Деревья"),
        ("Normal", "Для чего нужен щит?", "Для защиты от атак", "Для полёта", "Для еды", "Для крафта"),
        ("Normal", "Что происходит при попадании воды на лаву в некоторых условиях?", "Может образоваться обсидиан", "Всегда появляется алмаз", "Лава исчезает навсегда", "Появляется деревня"),
        ("Normal", "Что делает печь?", "Плавит и готовит", "Строит дом", "Телепортирует", "Вызывает дождь"),
        ("Normal", "Для чего нужен сундук?", "Для хранения предметов", "Для зачарования", "Для полёта", "Для добычи дерева"),
        ("Normal", "Что делает зачарованный предмет?", "Получает дополнительные свойства", "Становится едой", "Телепортируется", "Исчезает"),
        ("Normal", "Зачем нужны координаты?", "Чтобы ориентироваться в мире", "Чтобы крафтить еду", "Чтобы менять текстуры", "Чтобы включать музыку"),

        ("Hard", "Почему игроку важно подготовиться к Nether?", "Там опасные мобы и сложные условия", "Там нет блоков", "Там только деревья", "Там нельзя двигаться"),
        ("Hard", "Зачем строят фермы?", "Для автоматического или удобного получения ресурсов", "Для смены версии игры", "Для запуска сервера", "Для смены скина"),
        ("Hard", "Что полезно сделать перед дальним путешествием?", "Взять еду, инструменты и запасные ресурсы", "Взять только семена", "Оставить все вещи дома", "Идти без еды"),
        ("Hard", "Почему деревни полезны?", "В них есть жители, здания и ресурсы", "Они всегда дают алмазы", "Они открывают The End", "Они выключают мобов"),
        ("Hard", "Для чего нужен редстоун?", "Для создания механизмов и автоматизации", "Для еды", "Для брони", "Для сна"),
        ("Hard", "Почему освещение важно в выживании?", "Оно помогает контролировать появление враждебных мобов", "Оно увеличивает алмазы", "Оно ускоряет игрока", "Оно меняет биом"),
        ("Hard", "Что помогает пережить бой с сильным противником?", "Подготовленное снаряжение и стратегия", "Только еда", "Только факел", "Только дерево"),

        ("Expert", "Почему координаты особенно важны в Nether?", "Расстояния между измерениями масштабируются", "Там нет направлений", "Они нужны только для крафта", "Они отключают мобов"),
        ("Expert", "Что является хорошим принципом автоматической фермы?", "Стабильный источник, обработка и сбор результата", "Случайные блоки", "Только вода", "Только сундук"),
        ("Expert", "Почему резерв ресурсов важен перед походом?", "Ошибка или смерть может привести к потере времени и предметов", "Потому что инвентарь должен быть полным", "Потому что мобы исчезают", "Потому что игра требует 64 предмета"),
        ("Expert", "Что объединяет эффективное выживание и строительство?", "Планирование ресурсов и пространства", "Только скорость", "Только редстоун", "Только броня"),
        ("Expert", "Почему редстоун может сильно изменить игру?", "Он позволяет создавать сложные автоматизированные системы", "Он заменяет все руды", "Он делает игрока бессмертным", "Он открывает каждую дверь"),
        ("Expert", "Что помогает построить большой проект без хаоса?", "План, ресурсы и поэтапная работа", "Случайный крафт", "Отсутствие сундуков", "Только одна кирка"),
        ("Expert", "Как лучше готовиться к сложной экспедиции?", "Заранее определить цель, маршрут, ресурсы и запасной план", "Идти без предметов", "Брать только оружие", "Игнорировать координаты"),
    ],

    3: [
        ("Easy", "Что такое Roblox?", "Платформа с пользовательскими играми", "Одна игра про ферму", "Операционная система", "Браузер"),
        ("Easy", "Как называется инструмент для создания игр Roblox?", "Roblox Studio", "Minecraft Studio", "Steam Editor", "Game Maker Pro"),
        ("Easy", "Что такое obby?", "Паркур/полоса препятствий", "Шахта", "Меч", "Магазин"),
        ("Easy", "Можно ли найти разные жанры в Roblox?", "Да", "Нет", "Только хоррор", "Только спорт"),
        ("Easy", "Что обычно использует игрок для управления?", "Персонажа", "Только карту", "Только чат", "Только меню"),
        ("Easy", "Что можно создавать в Roblox Studio?", "Игры и игровые объекты", "Только музыку", "Только видео", "Только документы"),
        ("Easy", "Что такое сервер игры?", "Среда, где игроки взаимодействуют", "Текстовый файл", "Скин", "Кнопка"),
        ("Easy", "Что важно делать при общении онлайн?", "Соблюдать безопасность", "Публиковать пароль", "Давать личные данные", "Открывать неизвестные ссылки"),

        ("Normal", "Зачем нужен Roblox Studio?", "Для разработки и тестирования игр", "Для просмотра фильмов", "Для музыки", "Для установки Windows"),
        ("Normal", "Что может содержать пользовательская игра?", "Механики, карты и объекты", "Только один блок", "Только текст", "Только музыку"),
        ("Normal", "Почему важно проверять игру перед публикацией?", "Чтобы находить ошибки", "Чтобы получить новый ПК", "Чтобы изменить пароль", "Чтобы удалить аккаунт"),
        ("Normal", "Что такое игровой жанр?", "Тип или стиль игры", "Имя игрока", "Название сервера", "Вид аккаунта"),
        ("Normal", "Что помогает найти интересные игры?", "Описание, жанр и отзывы", "Случайный пароль", "Только аватар", "Только цвет интерфейса"),
        ("Normal", "Что важно для безопасного аккаунта?", "Надёжный пароль и осторожность", "Передача пароля друзьям", "Открытие всех ссылок", "Публикация личных данных"),
        ("Normal", "Что можно делать в скриптах Roblox?", "Создавать игровую логику", "Менять пароль Windows", "Рисовать документы", "Устанавливать драйвер"),
        ("Normal", "Что такое NPC?", "Неигровой персонаж", "Тип сервера", "Вид валюты", "Кнопка меню"),

        ("Hard", "Почему тестирование игры важно?", "Ошибки могут ломать механику или мешать игрокам", "Оно делает интернет быстрее", "Оно увеличивает RAM", "Оно меняет жанр"),
        ("Hard", "Что помогает удерживать игроков в хорошей игре?", "Понятная цель и интересный игровой цикл", "Только случайные кнопки", "Сложное меню", "Пустая карта"),
        ("Hard", "Зачем разделять игровые механики на системы?", "Так код легче понимать и изменять", "Чтобы увеличить размер файла", "Чтобы удалить UI", "Чтобы отключить сервер"),
        ("Hard", "Что важно учитывать при создании интерфейса?", "Понятность и удобство", "Только количество кнопок", "Только размер файла", "Только музыку"),
        ("Hard", "Почему описание игры важно?", "Оно помогает игроку понять, что его ждёт", "Оно запускает сервер", "Оно создаёт карту", "Оно выдаёт Robux"),
        ("Hard", "Что полезно делать после обнаружения бага?", "Воспроизвести, найти причину и исправить", "Игнорировать", "Удалить игру", "Сменить аккаунт"),
        ("Hard", "Что делает хороший игровой цикл?", "Связывает действие игрока с понятным результатом и следующей целью", "Только показывает меню", "Удаляет прогресс", "Выключает звук"),

        ("Expert", "Почему модульная структура кода полезна?", "Отдельные системы проще тестировать и изменять", "Она запрещает скрипты", "Она удаляет объекты", "Она уменьшает карту"),
        ("Expert", "Как лучше улучшать игру после первой версии?", "Собирать обратную связь, анализировать проблемы и делать итерации", "Сразу переписать всё случайно", "Не тестировать", "Удалить старую версию"),
        ("Expert", "Что помогает сбалансировать сложность игры?", "Тестирование и анализ поведения игроков", "Только мнение разработчика", "Случайные числа", "Только графика"),
        ("Expert", "Почему телеметрия может быть полезна разработчику?", "Она показывает, как игроки реально взаимодействуют с игрой", "Она автоматически создаёт карту", "Она заменяет код", "Она выключает сервер"),
        ("Expert", "Что является хорошей практикой при разработке?", "Небольшие изменения с регулярным тестированием", "Менять всё сразу без проверки", "Не сохранять версии", "Не читать ошибки"),
        ("Expert", "Что важнее всего при создании сложной системы?", "Чёткая структура, проверка и понятные данные", "Максимум случайности", "Отсутствие тестов", "Один огромный скрипт без структуры"),
        ("Expert", "Как можно использовать данные игроков?", "Чтобы находить проблемы и улучшать игровой опыт", "Чтобы узнать пароли", "Чтобы удалить аккаунты", "Чтобы отключить интернет"),
    ]
}

def make_question(difficulty, question, correct, *wrong):
    options = [correct] + list(wrong)
    random.shuffle(options)
    letters = "ABCD"
    return {
        "difficulty": difficulty,
        "question": question,
        "options": [f"{letters[i]}) {x}" for i, x in enumerate(options)],
        "answer": letters[options.index(correct)]
    }

quizzes = {
    gid: [make_question(item[0], item[1], item[2], item[3], item[4], item[5])
           for item in items]
    for gid, items in raw_quizzes.items()
}

# -----------------------------
# 🏆 LEVELS
# -----------------------------
levels = {
    1: 0,
    2: 50,
    3: 100,
    4: 200,
    5: 350,
    6: 500,
    7: 750,
    8: 1000
}

def calculate_level(xp):
    current = 1
    for lvl, required_xp in sorted(levels.items()):
        if xp >= required_xp:
            current = lvl
    return current

# -----------------------------
# 📊 PROGRESS
# -----------------------------
def empty_game_stats():
    return {
        "level": "Not tested",
        "last_score": 0,
        "quizzes": 0,
        "easy_correct": 0, "easy_total": 0,
        "normal_correct": 0, "normal_total": 0,
        "hard_correct": 0, "hard_total": 0,
        "expert_correct": 0, "expert_total": 0
    }

def default_progress():
    return {
        "xp": 0,
        "level": 1,
        "quizzes_completed": 0,
        "guides_opened": 0,
        "game_ml": {}
    }

progress = default_progress()

def get_game_statistics(game_number):
    key = str(game_number)
    if key not in progress["game_ml"] or not isinstance(progress["game_ml"][key], dict):
        progress["game_ml"][key] = empty_game_stats()

    stats = progress["game_ml"][key]
    defaults = empty_game_stats()
    for k, v in defaults.items():
        if k not in stats:
            stats[k] = v
    return stats

def save_progress():
    progress["level"] = calculate_level(progress.get("xp", 0))
    with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
        json.dump(progress, f, ensure_ascii=False, indent=2)

def load_progress():
    global progress
    if not os.path.exists(PROGRESS_FILE):
        progress = default_progress()
        return

    try:
        with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
            loaded = json.load(f)

        if not isinstance(loaded, dict):
            raise ValueError("progress.json должен содержать объект")

        base = default_progress()
        base.update(loaded)

        if not isinstance(base.get("game_ml"), dict):
            base["game_ml"] = {}

        progress = base
        progress["level"] = calculate_level(progress.get("xp", 0))

        for gid in games:
            get_game_statistics(gid)

    except Exception as e:
        print(f"⚠️ Не удалось загрузить progress.json: {e}")
        print("↪️ Создаю чистый прогресс.")
        progress = default_progress()

load_progress()

# -----------------------------
# 🤖 REAL ML HISTORY
# -----------------------------
def default_ml_history():
    return []

ml_history = default_ml_history()

def load_ml_history():
    global ml_history
    if not os.path.exists(ML_DATA_FILE):
        ml_history = []
        return

    try:
        with open(ML_DATA_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
        ml_history = data if isinstance(data, list) else []
    except Exception as e:
        print(f"⚠️ Не удалось загрузить ML history: {e}")
        ml_history = []

def save_ml_history():
    with open(ML_DATA_FILE, "w", encoding="utf-8") as f:
        json.dump(ml_history, f, ensure_ascii=False, indent=2)

load_ml_history()

ML_FEATURE_NAMES = [
    "score_percent",
    "quizzes",
    "xp",
    "easy",
    "normal",
    "hard",
    "expert",
    "correct_answers",
    "wrong_answers"
]

def percent(correct, total):
    return round((correct / total) * 100, 1) if total else 0.0

def get_feature_row(game_number, score_percent=None):
    stats = get_game_statistics(game_number)

    easy = percent(stats["easy_correct"], stats["easy_total"])
    normal = percent(stats["normal_correct"], stats["normal_total"])
    hard = percent(stats["hard_correct"], stats["hard_total"])
    expert = percent(stats["expert_correct"], stats["expert_total"])

    correct_answers = (
        stats["easy_correct"]
        + stats["normal_correct"]
        + stats["hard_correct"]
        + stats["expert_correct"]
    )
    total_answers = (
        stats["easy_total"]
        + stats["normal_total"]
        + stats["hard_total"]
        + stats["expert_total"]
    )

    if score_percent is None:
        score_percent = stats["last_score"]

    return [
        float(score_percent),
        int(stats["quizzes"]),
        int(progress["xp"]),
        float(easy),
        float(normal),
        float(hard),
        float(expert),
        int(correct_answers),
        int(total_answers - correct_answers)
    ]

def skill_label(score):
    if score >= 85:
        return "Expert"
    if score >= 70:
        return "Advanced"
    if score >= 50:
        return "Intermediate"
    return "Beginner"

def record_ml_result(game_number, score_percent):
    """
    Сохраняем РЕАЛЬНЫЙ результат игрока.
    Target — фактически достигнутый skill label по результату викторины.
    """
    row = get_feature_row(game_number, score_percent)
    record = {
        "game": game_number,
        "features": dict(zip(ML_FEATURE_NAMES, row)),
        "target": skill_label(score_percent)
    }

    ml_history.append(record)
    save_ml_history()
    return record

# -----------------------------
# 💾 ML MODEL
# -----------------------------
def save_ml_model(model):
    try:
        joblib.dump(model, ML_MODEL_FILE)
        return True
    except Exception as e:
        print(f"⚠️ Ошибка сохранения ML-модели: {e}")
        return False

def load_ml_model():
    if not os.path.exists(ML_MODEL_FILE):
        return None
    try:
        return joblib.load(ML_MODEL_FILE)
    except Exception as e:
        print(f"⚠️ Ошибка загрузки ML-модели: {e}")
        return None

def train_ml_from_real_data(verbose=True):
    """
    MAX:
    - если реальных данных мало, сохраняем существующую модель;
    - когда данных достаточно и есть минимум 2 класса,
      обучаем модель заново на РЕАЛЬНЫХ результатах.
    """
    if not SKLEARN_OK:
        if verbose:
            print("⚠️ scikit-learn не установлен.")
        return load_ml_model()

    if len(ml_history) < 4:
        if verbose:
            print(f"🧠 Real ML data: {len(ml_history)} result(s). Нужно минимум 4.")
        return load_ml_model()

    X = [list(item["features"].values()) for item in ml_history]
    y = [item["target"] for item in ml_history]

    if len(set(y)) < 2:
        if verbose:
            print("🧠 Реальных данных пока недостаточно для нескольких классов.")
        return load_ml_model()

    model = DecisionTreeClassifier(max_depth=5, random_state=42)
    model.fit(X, y)

    if save_ml_model(model):
        if verbose:
            print(f"🤖 ML retrained on {len(ml_history)} real result(s).")
    return model

# -----------------------------
# 🎯 ADAPTIVE DIFFICULTY
# -----------------------------
DIFFICULTIES = ["Easy", "Normal", "Hard", "Expert"]

def difficulty_percent(stats, difficulty):
    return percent(
        stats[f"{difficulty.lower()}_correct"],
        stats[f"{difficulty.lower()}_total"]
    )

def choose_adaptive_difficulty(game_number):
    stats = get_game_statistics(game_number)
    tested = [
        d for d in DIFFICULTIES
        if stats[f"{d.lower()}_total"] > 0
    ]

    if not tested:
        return "Easy"

    weakest = min(tested, key=lambda d: difficulty_percent(stats, d))

    if difficulty_percent(stats, weakest) < 50:
        return weakest

    if "Expert" in tested and difficulty_percent(stats, "Expert") >= 80:
        return "Expert"

    if "Hard" in tested and difficulty_percent(stats, "Hard") >= 75:
        return "Expert"

    if "Normal" in tested and difficulty_percent(stats, "Normal") >= 75:
        return "Hard"

    return "Normal"

# -----------------------------
# 🔀 QUESTION SHUFFLE
# -----------------------------
def shuffle_question(question):
    pairs = list(zip(question["options"], question["answer"]))
    # Rebuild from displayed text while preserving correct answer text.
    correct_text = next(
        option[3:] for option in question["options"]
        if option.startswith(question["answer"] + ") ")
    )
    texts = [option[3:] for option in question["options"]]
    random.shuffle(texts)

    letters = "ABCD"
    new_options = [f"{letters[i]}) {texts[i]}" for i in range(4)]
    new_answer = letters[texts.index(correct_text)]

    return {
        "difficulty": question["difficulty"],
        "question": question["question"],
        "options": new_options,
        "answer": new_answer
    }

# -----------------------------
# 🧠 ADAPTIVE QUIZ
# -----------------------------
def adaptive_quiz(game_number, amount=5):
    if game_number not in games:
        print("❌ Неверный номер игры.")
        return None

    stats = get_game_statistics(game_number)
    difficulty = choose_adaptive_difficulty(game_number)

    pool = [
        q for q in quizzes[game_number]
        if q["difficulty"] == difficulty
    ]

    if not pool:
        print("❌ Нет вопросов для этой сложности.")
        return None

    selected = random.sample(pool, min(amount, len(pool)))
    correct_count = 0

    print(f"\n🎮 {games[game_number]['name']}")
    print(f"🎯 Adaptive difficulty: {difficulty}")
    print("-" * 50)

    for index, original in enumerate(selected, 1):
        q = shuffle_question(original)

        print(f"\n{index}. {q['question']}")
        for option in q["options"]:
            print(option)

        while True:
            answer = input("Твой ответ (A/B/C/D): ").strip().upper()
            if answer in "ABCD":
                break
            print("⚠️ Введи A, B, C или D.")

        key_correct = f"{difficulty.lower()}_correct"
        key_total = f"{difficulty.lower()}_total"

        stats[key_total] += 1

        if answer == q["answer"]:
            correct_count += 1
            stats[key_correct] += 1
            print("✅ Correct!")
        else:
            print(f"❌ Wrong! Правильный ответ: {q['answer']}")

    score_percent = round((correct_count / len(selected)) * 100, 1)
    xp_gain = correct_count * 10

    stats["last_score"] = score_percent
    stats["quizzes"] += 1
    stats["level"] = skill_label(score_percent)

    progress["xp"] += xp_gain
    progress["quizzes_completed"] += 1
    progress["level"] = calculate_level(progress["xp"])

    # Real ML data is saved AFTER the result becomes part of the player's history.
    record = record_ml_result(game_number, score_percent)

    # Train on real history when possible.
    model = train_ml_from_real_data(verbose=True)

    save_progress()

    print("\n" + "=" * 50)
    print(f"🏁 Result: {correct_count}/{len(selected)} ({score_percent}%)")
    print(f"⭐ XP +{xp_gain}")
    print(f"🏆 Global level: {progress['level']}")
    print(f"🧠 Real ML records: {len(ml_history)}")
    print(f"🎯 Observed skill: {record['target']}")
    print("=" * 50)

    return {
        "game": game_number,
        "difficulty": difficulty,
        "correct": correct_count,
        "total": len(selected),
        "score": score_percent,
        "xp_gain": xp_gain,
        "skill": record["target"]
    }

# -----------------------------
# 🔎 WEAK DIFFICULTY / TOPIC
# -----------------------------
def find_weak_difficulty(game_number):
    stats = get_game_statistics(game_number)
    tested = [
        d for d in DIFFICULTIES
        if stats[f"{d.lower()}_total"] > 0
    ]

    if not tested:
        return None

    return min(tested, key=lambda d: difficulty_percent(stats, d))

def guide_for_weak_difficulty(game_number):
    weak = find_weak_difficulty(game_number)

    if weak is None:
        return None

    mapping = {
        1: {"Easy": 8, "Normal": 4, "Hard": 5, "Expert": 6},
        2: {"Easy": 1, "Normal": 3, "Hard": 6, "Expert": 7},
        3: {"Easy": 1, "Normal": 2, "Hard": 6, "Expert": 3}
    }

    return mapping[game_number].get(weak)

def topic_analyzer(game_number):
    weak = find_weak_difficulty(game_number)

    if weak is None:
        return None

    topics = {
        1: {
            "Easy": "Основы",
            "Normal": "Персонажи и монстры",
            "Hard": "Головоломки",
            "Expert": "Лор Playtime Co."
        },
        2: {
            "Easy": "Выживание",
            "Normal": "Добыча руд",
            "Hard": "Nether",
            "Expert": "The End"
        },
        3: {
            "Easy": "Основы",
            "Normal": "Поиск игр",
            "Hard": "Создание игры",
            "Expert": "Roblox Studio"
        }
    }

    return topics[game_number].get(weak)

# -----------------------------
# 🤖 ML RECOMMENDATION
# -----------------------------
def run_ml_v2(game_number):
    stats = get_game_statistics(game_number)

    score_percent = stats["last_score"]
    easy = difficulty_percent(stats, "Easy")
    normal = difficulty_percent(stats, "Normal")
    hard = difficulty_percent(stats, "Hard")
    expert = difficulty_percent(stats, "Expert")

    # Prefer fresh real-data training when possible.
    model = train_ml_from_real_data(verbose=False)

    if model is None and SKLEARN_OK:
        # Small fallback seed model so MAX works before enough real results exist.
        X = [
            [95, 10, 1000, 95, 90, 90, 90, 38, 2],
            [85, 8, 750, 85, 80, 80, 70, 30, 10],
            [70, 5, 500, 80, 70, 60, 40, 20, 5],
            [55, 3, 250, 70, 50, 40, 20, 12, 8],
            [35, 1, 100, 50, 30, 20, 0, 5, 10],
            [20, 1, 50, 30, 20, 10, 0, 2, 13]
        ]
        y = ["Expert", "Advanced", "Advanced", "Intermediate", "Beginner", "Beginner"]

        model = DecisionTreeClassifier(max_depth=5, random_state=42)
        model.fit(X, y)
        save_ml_model(model)

    if model is None:
        print("⚠️ ML недоступен.")
        return None

    features = [[
        score_percent,
        stats["quizzes"],
        progress["xp"],
        easy,
        normal,
        hard,
        expert,
        stats["easy_correct"]
        + stats["normal_correct"]
        + stats["hard_correct"]
        + stats["expert_correct"],
        stats["easy_total"]
        + stats["normal_total"]
        + stats["hard_total"]
        + stats["expert_total"]
        - (
            stats["easy_correct"]
            + stats["normal_correct"]
            + stats["hard_correct"]
            + stats["expert_correct"]
        )
    ]]

    prediction = model.predict(features)[0]

    print("\n🤖 ML Recommendation")
    print(f"Game: {games[game_number]['name']}")
    print(f"Prediction: {prediction}")
    print(f"Weak difficulty: {find_weak_difficulty(game_number) or 'Not tested'}")
    print(f"Weak topic: {topic_analyzer(game_number) or 'Not enough data'}")

    guide_id = guide_for_weak_difficulty(game_number)
    if guide_id:
        print(f"📚 Recommended guide: {guide_id}. {guides[game_number]['guides'][guide_id]}")
    else:
        print("📚 Recommended guide: complete a quiz first.")

    return prediction

# -----------------------------
# 📚 GUIDES UI
# -----------------------------
def show_games():
    print("\n🎮 GAMES")
    for gid, game in games.items():
        print(f"{gid}. {game['name']} — {game['description']}")

def show_guides(game_number=None):
    if game_number is None:
        show_games()
        try:
            game_number = int(input("Выбери игру: ").strip())
        except ValueError:
            print("❌ Нужен номер игры.")
            return

    if game_number not in guides:
        print("❌ Неверный номер игры.")
        return

    print(f"\n📚 {guides[game_number]['name']} — GUIDES")
    for gid, title in guides[game_number]["guides"].items():
        print(f"{gid}. {title}")

    try:
        guide_id = int(input("Выбери гайд (0 = назад): ").strip())
    except ValueError:
        print("❌ Нужен номер.")
        return

    if guide_id == 0:
        return

    title = guides[game_number]["guides"].get(guide_id)
    if title is None:
        print("❌ Такого гайда нет.")
        return

    progress["guides_opened"] += 1
    save_progress()

    print(f"\n📖 {title}")
    print("ℹ️ В MAX эта система уже готова к расширению полноценными текстами гайдов.")
    print("💡 Сначала анализируем слабую тему, затем открываем нужный гайд.")

# -----------------------------
# 📊 PROGRESS UI
# -----------------------------
def show_progress():
    print("\n📊 PROGRESS")
    print(f"⭐ XP: {progress['xp']}")
    print(f"🏆 Level: {progress['level']}")
    print(f"🧠 Quizzes completed: {progress['quizzes_completed']}")
    print(f"📚 Guides opened: {progress['guides_opened']}")
    print(f"🤖 Real ML records: {len(ml_history)}")

    for gid, game in games.items():
        stats = get_game_statistics(gid)
        print(f"\n🎮 {game['name']}")
        print(f"   Level: {stats['level']}")
        print(f"   Last score: {stats['last_score']}%")
        print(f"   Quizzes: {stats['quizzes']}")

        for d in DIFFICULTIES:
            total = stats[f"{d.lower()}_total"]
            if total:
                print(f"   {d}: {difficulty_percent(stats, d)}%")

# -----------------------------
# 🔁 FULL LEARNING LOOP
# -----------------------------
def learning_loop(game_number, quiz_amount=5):
    """
    MAX loop:
    Quiz → real result → ML → weak difficulty → weak topic → guide → next quiz
    """
    result = adaptive_quiz(game_number, quiz_amount)
    if result is None:
        return

    print("\n🧠 ANALYSIS")
    weak = find_weak_difficulty(game_number)
    topic = topic_analyzer(game_number)
    guide_id = guide_for_weak_difficulty(game_number)

    print(f"🎯 Weak difficulty: {weak or '—'}")
    print(f"🧩 Weak topic: {topic or '—'}")

    if guide_id:
        print(f"📚 Next guide: {guide_id}. {guides[game_number]['guides'][guide_id]}")

    run_ml_v2(game_number)

# -----------------------------
# 🧪 SELF CHECK
# -----------------------------
def self_check():
    print("\n🧪 GAME HELPER MAX SELF-CHECK")
    print("-" * 50)

    assert len(games) == 3, "Games count error"
    assert sum(len(g["guides"]) for g in guides.values()) == 30, "Guides count error"
    assert sum(len(qs) for qs in quizzes.values()) == 90, "Quiz count error"

    expected = {1: 30, 2: 30, 3: 30}
    for gid in games:
        assert len(quizzes[gid]) == expected[gid], f"Quiz count error for game {gid}"

    for gid in games:
        counts = {d: 0 for d in DIFFICULTIES}
        for q in quizzes[gid]:
            counts[q["difficulty"]] += 1
            assert len(q["options"]) == 4, "Question must have 4 options"
            assert q["answer"] in "ABCD", "Answer must be A/B/C/D"

        assert counts == {"Easy": 8, "Normal": 8, "Hard": 7, "Expert": 7}, \
            f"Difficulty distribution error for game {gid}"

    for gid in games:
        stats = get_game_statistics(gid)
        assert all(k in stats for k in empty_game_stats()), "Stats fields missing"

    print("✅ Games: 3")
    print("✅ Guides: 30")
    print("✅ Questions: 90")
    print("✅ Distribution: 8/8/7/7 per game")
    print("✅ Progress system")
    print("✅ Adaptive Quiz")
    print("✅ Real ML history")
    print("✅ ML model persistence")
    print("✅ ML Recommendation")
    print("✅ Topic Analyzer")
    print("✅ Learning Loop")
    print("🎉 GAME HELPER MAX CHECK PASSED!")

# -----------------------------
# 🧭 MAIN MENU
# -----------------------------
def main_menu():
    while True:
        print("\n" + "=" * 55)
        print("🎮 GAME HELPER MAX")
        print("=" * 55)
        print("1. 🎮 Игры")
        print("2. 📚 Гайды")
        print("3. 🧠 Adaptive Quiz")
        print("4. 🤖 ML Recommendation")
        print("5. 🧠 Topic Analyzer")
        print("6. 🔁 Full Learning Loop")
        print("7. 📊 Прогресс")
        print("8. 🧪 Self Check")
        print("9. 🚪 Выход")

        choice = input("\nВыбери: ").strip()

        if choice == "1":
            show_games()

        elif choice == "2":
            show_guides()

        elif choice == "3":
            show_games()
            try:
                gid = int(input("Выбери игру: ").strip())
                adaptive_quiz(gid, 5)
            except ValueError:
                print("❌ Нужен номер игры.")

        elif choice == "4":
            show_games()
            try:
                gid = int(input("Выбери игру: ").strip())
                run_ml_v2(gid)
            except ValueError:
                print("❌ Нужен номер игры.")

        elif choice == "5":
            show_games()
            try:
                gid = int(input("Выбери игру: ").strip())
                topic = topic_analyzer(gid)
                print(f"🧠 Weak topic: {topic or 'Not enough data'}")
            except ValueError:
                print("❌ Нужен номер игры.")

        elif choice == "6":
            show_games()
            try:
                gid = int(input("Выбери игру: ").strip())
                learning_loop(gid, 5)
            except ValueError:
                print("❌ Нужен номер игры.")

        elif choice == "7":
            show_progress()

        elif choice == "8":
            self_check()

        elif choice == "9":
            print("👋 Bye!")
            break

        else:
            print("❌ Неверный пункт.")

# -----------------------------
# 🚀 STARTUP
# -----------------------------
print("🚀 GAME HELPER MAX READY!")
print(f"🎮 Games: {len(games)}")
print(f"📚 Guides: {sum(len(g['guides']) for g in guides.values())}")
print(f"🧠 Questions: {sum(len(qs) for qs in quizzes.values())}")
print(f"🤖 scikit-learn: {'READY' if SKLEARN_OK else 'NOT INSTALLED'}")
print(f"💾 ML history: {len(ml_history)} real result(s)")
print(f"💾 ML model: {'FOUND' if os.path.exists(ML_MODEL_FILE) else 'WILL BE CREATED'}")
print(f"📊 Progress: Level {progress['level']} | XP {progress['xp']}")
print("\n💡 Запусти self_check() для полной проверки.")
print("💡 Запусти main_menu() для Game Helper.")



# ============================================================
# 🤖 GAME HELPER TELEGRAM MAX
# ============================================================

import asyncio
from telegram import InlineKeyboardButton, InlineKeyboardMarkup, Update
from telegram.ext import Application, CallbackQueryHandler, CommandHandler, ContextTypes

from getpass import getpass

BOT_TOKEN = getpass("🔐 Введи Telegram Bot Token: ").strip()
TELEGRAM_USERS_FILE = "telegram_users.json"

def load_telegram_users():
    if not os.path.exists(TELEGRAM_USERS_FILE):
        return {}
    try:
        with open(TELEGRAM_USERS_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except Exception:
        return {}

telegram_users = load_telegram_users()

def save_telegram_users():
    with open(TELEGRAM_USERS_FILE, "w", encoding="utf-8") as f:
        json.dump(telegram_users, f, ensure_ascii=False, indent=2)

def get_user_state(user_id):
    key = str(user_id)
    if key not in telegram_users:
        telegram_users[key] = {
            "game": None,
            "quiz_difficulty": None,
            "quiz_questions": [],
            "quiz_index": 0,
            "quiz_correct": 0,
            "quiz_started": False
        }
        save_telegram_users()
    return telegram_users[key]

def главное_меню():
    return InlineKeyboardMarkup([
        [InlineKeyboardButton("🎮 Игры", callback_data="игры")],
        [InlineKeyboardButton("📚 Гайды", callback_data="гайды")],
        [InlineKeyboardButton("🧠 Викторина", callback_data="викторина")],
        [InlineKeyboardButton("🤖 ML Анализ", callback_data="ml")],
        [InlineKeyboardButton("🧩 Слабая тема", callback_data="тема")],
        [InlineKeyboardButton("📊 Прогресс", callback_data="прогресс")]
    ])

def клавиатура_игр(префикс="игра"):
    rows = [
        [InlineKeyboardButton(
            f"🎮 {game['name']}",
            callback_data=f"{префикс}:{gid}"
        )]
        for gid, game in games.items()
    ]
    rows.append([InlineKeyboardButton("⬅️ Главное меню", callback_data="меню")])
    return InlineKeyboardMarkup(rows)

async def команда_start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    get_user_state(update.effective_user.id)
    await update.message.reply_text(
        "🎮 <b>GAME HELPER MAX</b>\n\n"
        "Добро пожаловать!\n\n"
        "🎮 Игры\n"
        "📚 Гайды\n"
        "🧠 Адаптивная викторина\n"
        "🤖 ML-анализ\n"
        "⭐ XP и уровни\n"
        "📊 Прогресс\n\n"
        "Выбирай действие:",
        parse_mode="HTML",
        reply_markup=главное_меню()
    )

async def показать_игры(query):
    await query.edit_message_text(
        "🎮 <b>ИГРЫ</b>\n\nВыбери игру:",
        parse_mode="HTML",
        reply_markup=клавиатура_игр()
    )

async def показать_игру(query, game_number):
    game = games[game_number]
    keyboard = [
        [InlineKeyboardButton("🧠 Викторина", callback_data=f"квиз:{game_number}")],
        [InlineKeyboardButton("📚 Гайды", callback_data=f"гайды_игры:{game_number}")],
        [InlineKeyboardButton("🤖 ML Анализ", callback_data=f"мл_игры:{game_number}")],
        [InlineKeyboardButton("🧩 Слабая тема", callback_data=f"тема_игры:{game_number}")],
        [InlineKeyboardButton("⬅️ Игры", callback_data="игры")]
    ]
    await query.edit_message_text(
        f"🎮 <b>{game['name']}</b>\n\n{game['description']}\n\nЧто делаем?",
        parse_mode="HTML",
        reply_markup=InlineKeyboardMarkup(keyboard)
    )

async def показать_гайды(query):
    await query.edit_message_text(
        "📚 <b>ГАЙДЫ</b>\n\nВыбери игру:",
        parse_mode="HTML",
        reply_markup=клавиатура_игр("гайды_игры")
    )

async def показать_список_гайдов(query, game_number):
    rows = [
        [InlineKeyboardButton(
            f"{gid}. {title}",
            callback_data=f"гайд:{game_number}:{gid}"
        )]
        for gid, title in guides[game_number]["guides"].items()
    ]
    rows.append([InlineKeyboardButton("⬅️ Назад", callback_data="гайды")])
    await query.edit_message_text(
        f"📚 <b>{games[game_number]['name']}</b>\n\nВыбери гайд:",
        parse_mode="HTML",
        reply_markup=InlineKeyboardMarkup(rows)
    )

GUIDE_TEXTS = {1: {1: '🎯 Цель: пройти фабрику Playtime Co. и получить GrabPack.\n\n🧩 Шаги: исследуй стартовую зону, найди GrabPack, используй цветные руки для механизмов и следуй подсказкам фабрики.\n\n💡 Совет: проверяй комнаты внимательно и не спеши.', 2: '🎯 Цель: пройти Chapter 2 и использовать новые возможности GrabPack.\n\n🧩 Исследуй Game Station, проходи игровые испытания и решай головоломки.\n\n💡 Совет: запоминай порядок действий в сложных головоломках.', 3: '🎯 Цель: исследовать Playcare и продвинуться через опасные зоны.\n\n🧩 Собирай необходимые предметы, изучай локации и следи за окружением.\n\n💡 Совет: сначала изучи безопасную часть новой зоны.', 4: '👾 Изучи основных персонажей и монстров Poppy Playtime.\n\n🧠 Обращай внимание на их поведение, появление и роль в событиях.\n\n💡 Совет: знание поведения врага помогает правильно выбирать момент для движения.', 5: '🧩 Основной принцип головоломок: найди интерактивные объекты и определи правильный порядок действий.\n\n💡 Совет: проверь питание, цветовые подсказки, переключатели и связанные механизмы.', 6: '🔎 Ищи скрытые предметы, необычные комнаты, записи и Easter Eggs.\n\n💡 Совет: после прохождения можно вернуться и внимательно исследовать знакомые места.', 7: '📖 Изучай историю Playtime Co. через записи, VHS-кассеты и детали окружения.\n\n🧠 Старайся расположить события в хронологическом порядке.\n\n💡 Совет: не пропускай записи и подсказки.', 8: '🎮 Новичку: исследуй спокойно, читай подсказки и проверяй интерактивные предметы.\n\n💡 Совет: если застрял, вернись на несколько шагов назад и проверь, что пропустил.'}, 2: {1: '⛏️ Первые 10 минут: собери дерево, сделай верстак, деревянные инструменты и добудь камень.\n\n🎯 Найди еду и подготовь место для убежища.\n\n💡 Совет: не трать первый день на большой дом.', 2: '🌙 Перед первой ночью подготовь убежище, факелы и еду.\n\n🛡️ Закрой вход и держи инструменты рядом.\n\n💡 Совет: маленькое безопасное убежище лучше красивого недостроенного дома.', 3: '⛏️ Для добычи руд используй подходящую кирку и освещай шахты.\n\n🧭 Проверяй пещеры и боковые тоннели.\n\n💡 Совет: не копай прямо вниз и всегда учитывай лаву.', 4: '💎 Алмазы находятся глубоко под землёй.\n\n⛏️ Используй подходящую кирку, освещай туннели и проверяй открытые пещеры.\n\n💡 Совет: носи с собой блоки для безопасного обхода лавы.', 5: '🛡️ Для защиты используй щит, броню и оружие.\n\n⚔️ Улучшай снаряжение постепенно и следи за прочностью.\n\n💡 Совет: железное снаряжение — важный шаг перед опасными путешествиями.', 6: '🔥 Nether — опасное измерение с лавой, крепостями и уникальными ресурсами.\n\n🎒 Возьми еду, блоки, броню и запасные инструменты.\n\n💡 Совет: отметь координаты портала и не уходи слишком далеко без подготовки.', 7: '🐉 The End — финальное измерение с главным сражением.\n\n🎯 Подготовь броню, оружие, еду и предметы для выживания.\n\n💡 Совет: перед сражением изучи арену и подготовь безопасные места.', 8: '🏠 Начинай строительство с основания, стен, входа и освещения.\n\n🎨 После функциональности добавляй декор и детали.\n\n💡 Совет: используй несколько подходящих блоков вместо одного материала.', 9: '✨ Зачарования усиливают предметы.\n\n📚 Собери стол зачарований и опыт, затем выбирай эффекты под задачу.\n\n💡 Совет: не трать лучшие ресурсы на случайные зачарования.', 10: '🌾 Фермы помогают получать ресурсы регулярно.\n\n🎯 Начни с простой фермы еды или животных, затем переходи к автоматизации.\n\n💡 Совет: сначала сделай ферму надёжной, потом усложняй её.', 11: '👾 Мобы бывают мирными, нейтральными и враждебными.\n\n🧠 Изучай условия их появления и поведение.\n\n💡 Совет: освещение и безопасная дистанция часто решают проблему.', 12: '🏘️ Деревни дают жителей, торговлю и полезные ресурсы.\n\n🛡️ Защити деревню и организуй безопасную территорию.\n\n💡 Совет: торговля может сильно ускорить развитие.', 13: '🔴 Редстоун позволяет создавать механизмы и автоматические системы.\n\n⚙️ Начни с рычага, кнопки, редстоун-пыли и простого поршня.\n\n💡 Совет: тестируй схему маленькими частями.', 14: '🧭 Выживание строится вокруг еды, убежища, инструментов и безопасности.\n\n🎯 Цепочка развития: дом → железо → ферма → Nether → The End.\n\n💡 Совет: ставь небольшие цели вместо попытки сделать всё сразу.'}, 3: {1: '🎮 Roblox — платформа с множеством пользовательских игр.\n\n🧭 Изучи поиск, управление и настройки аккаунта.\n\n💡 Совет: сначала попробуй несколько разных жанров.', 2: '🔎 Чтобы найти подходящую игру, смотри описание, жанр, рейтинг и активность игроков.\n\n💡 Совет: не суди игру только по обложке — проверь описание и отзывы.', 3: '🛠️ Roblox Studio используется для создания игр.\n\n🎯 Начни с простой сцены, объектов и базовой логики.\n\n💡 Совет: сначала сделай маленький рабочий прототип.', 4: '🔐 Безопасность: используй сильный пароль и не передавай его другим.\n\n🛡️ Не открывай подозрительные ссылки и не сообщай личные данные незнакомцам.\n\n💡 Совет: при сомнении лучше выйти из чата или обратиться к взрослому.', 5: '🎮 В Roblox есть obby, horror, adventure, tycoon, simulator и другие жанры.\n\n💡 Совет: выбирай жанр по своему интересу и пробуй разные варианты.', 6: '🛠️ Создание игры начинается с идеи и простой механики.\n\n🎯 Сделай маленькую карту, цель игрока и тестовый уровень.\n\n💡 Совет: сначала добейся, чтобы основная механика работала.', 7: '🚀 Новичку лучше изучать одну игру или механику за раз.\n\n🧠 Читай подсказки, экспериментируй и сохраняй прогресс.\n\n💡 Совет: не пытайся освоить всё Roblox одновременно.', 8: '❓ FAQ: если игра не запускается, проверь интернет, обновление приложения и настройки устройства.\n\n💡 Для проблем конкретной игры сначала проверь её описание и инструкции разработчика.\n\n💡 Совет: если проблема повторяется, перезапусти игру и проверь соединение.'}}

async def показать_гайд(query, game_number, guide_id):
    title = guides[game_number]["guides"].get(guide_id)
    if not title:
        await query.answer("Гайд не найден.")
        return

    progress["guides_opened"] += 1
    save_progress()
    guide_text = GUIDE_TEXTS.get(game_number, {}).get(guide_id, "ℹ️ Текст гайда пока не добавлен.")

    await query.edit_message_text(
        f"📖 <b>{title}</b>\n\n"
        f"🎮 {games[game_number]['name']}\n\n"
        f"{guide_text}\n\n"
        "🎯 После гайда можно сразу пройти викторину.",
        parse_mode="HTML",
        reply_markup=InlineKeyboardMarkup([
            [InlineKeyboardButton("🧠 Пройти викторину", callback_data=f"квиз:{game_number}")],
            [InlineKeyboardButton("⬅️ К гайдам", callback_data=f"гайды_игры:{game_number}")],
            [InlineKeyboardButton("🏠 Меню", callback_data="меню")]
        ])
    )

async def начать_викторину(query, game_number):
    state = get_user_state(query.from_user.id)
    difficulty = choose_adaptive_difficulty(game_number)
    pool = [q for q in quizzes[game_number] if q["difficulty"] == difficulty]

    if not pool:
        await query.answer("Нет вопросов для этой сложности.")
        return

    state.update({
        "game": game_number,
        "quiz_difficulty": difficulty,
        "quiz_questions": random.sample(pool, min(5, len(pool))),
        "quiz_index": 0,
        "quiz_correct": 0,
        "quiz_started": True
    })
    save_telegram_users()
    await отправить_вопрос(query)

async def отправить_вопрос(query):
    state = get_user_state(query.from_user.id)
    if not state["quiz_started"]:
        await query.answer("Викторина не запущена.")
        return

    index = state["quiz_index"]
    questions = state["quiz_questions"]

    if index >= len(questions):
        await завершить_викторину(query)
        return

    question = shuffle_question(questions[index])
    state["current_question"] = question
    save_telegram_users()

    keyboard = [
        [
            InlineKeyboardButton("A", callback_data="ответ:A"),
            InlineKeyboardButton("B", callback_data="ответ:B")
        ],
        [
            InlineKeyboardButton("C", callback_data="ответ:C"),
            InlineKeyboardButton("D", callback_data="ответ:D")
        ],
        [InlineKeyboardButton("❌ Выйти", callback_data="выйти_квиз")]
    ]

    text = (
        f"🧠 <b>ВИКТОРИНА</b>\n\n"
        f"🎮 {games[state['game']]['name']}\n"
        f"🎯 Сложность: <b>{state['quiz_difficulty']}</b>\n"
        f"❓ Вопрос {index + 1}/{len(questions)}\n\n"
        f"<b>{question['question']}</b>\n\n"
        + "\n".join(question["options"])
    )

    await query.edit_message_text(
        text,
        parse_mode="HTML",
        reply_markup=InlineKeyboardMarkup(keyboard)
    )

async def обработать_ответ(query, answer):
    state = get_user_state(query.from_user.id)
    if not state["quiz_started"] or "current_question" not in state:
        await query.answer("Викторина уже закончена.")
        return

    question = state["current_question"]
    difficulty = state["quiz_difficulty"]
    stats = get_game_statistics(state["game"])

    stats[f"{difficulty.lower()}_total"] += 1

    if answer == question["answer"]:
        state["quiz_correct"] += 1
        stats[f"{difficulty.lower()}_correct"] += 1
        result = "✅ <b>Правильно!</b>"
    else:
        result = f"❌ <b>Неправильно.</b>\nПравильный ответ: <b>{question['answer']}</b>"

    state["quiz_index"] += 1
    save_progress()
    save_telegram_users()

    await query.edit_message_text(
        f"{result}\n\nСчёт: {state['quiz_correct']}/{state['quiz_index']}",
        parse_mode="HTML",
        reply_markup=InlineKeyboardMarkup([
            [InlineKeyboardButton("➡️ Следующий вопрос", callback_data="следующий_вопрос")]
        ])
    )

async def завершить_викторину(query):
    state = get_user_state(query.from_user.id)
    game_number = state["game"]
    difficulty = state["quiz_difficulty"]
    total = len(state["quiz_questions"])
    correct = state["quiz_correct"]

    score = round(correct / total * 100, 1) if total else 0
    xp_gain = correct * 10

    stats = get_game_statistics(game_number)
    stats["last_score"] = score
    stats["quizzes"] += 1
    stats["level"] = skill_label(score)

    progress["xp"] += xp_gain
    progress["quizzes_completed"] += 1
    progress["level"] = calculate_level(progress["xp"])

    record_ml_result(game_number, score)
    train_ml_from_real_data(verbose=False)
    save_progress()

    state["quiz_started"] = False
    state.pop("current_question", None)
    save_telegram_users()

    weak = find_weak_difficulty(game_number)
    topic = topic_analyzer(game_number)
    guide_id = guide_for_weak_difficulty(game_number)

    text = (
        f"🏁 <b>ВИКТОРИНА ЗАКОНЧЕНА</b>\n\n"
        f"🎮 {games[game_number]['name']}\n"
        f"🎯 Сложность: {difficulty}\n"
        f"📊 Результат: <b>{correct}/{total} ({score}%)</b>\n"
        f"⭐ XP: +{xp_gain}\n"
        f"🏆 Уровень: {progress['level']}\n\n"
        f"🎯 Слабая сложность: {weak or '—'}\n"
        f"🧩 Слабая тема: {topic or '—'}"
    )

    if guide_id:
        text += f"\n📚 Рекомендуемый гайд:\n{guides[game_number]['guides'][guide_id]}"

    await query.edit_message_text(
        text,
        parse_mode="HTML",
        reply_markup=InlineKeyboardMarkup([
            [InlineKeyboardButton("🧠 Ещё раз", callback_data=f"квиз:{game_number}")],
            [InlineKeyboardButton("🤖 ML Анализ", callback_data=f"мл_игры:{game_number}")],
            [InlineKeyboardButton("📊 Прогресс", callback_data="прогресс")],
            [InlineKeyboardButton("🏠 Меню", callback_data="меню")]
        ])
    )

async def показать_ml(query, game_number):
    stats = get_game_statistics(game_number)

    if stats["quizzes"] == 0:
        text = (
            f"🤖 <b>ML АНАЛИЗ</b>\n\n"
            f"🎮 {games[game_number]['name']}\n\n"
            "Пока нет результатов.\nПройди викторину."
        )
    else:
        prediction = run_ml_v2(game_number)
        weak = find_weak_difficulty(game_number)
        topic = topic_analyzer(game_number)
        guide_id = guide_for_weak_difficulty(game_number)

        text = (
            f"🤖 <b>ML АНАЛИЗ</b>\n\n"
            f"🎮 {games[game_number]['name']}\n"
            f"🧠 Предсказание: <b>{prediction or '—'}</b>\n"
            f"🎯 Слабая сложность: <b>{weak or '—'}</b>\n"
            f"🧩 Слабая тема: <b>{topic or '—'}</b>\n"
            f"📊 Викторин: {stats['quizzes']}\n"
            f"💾 ML-записей: {len(ml_history)}"
        )

        if guide_id:
            text += f"\n\n📚 Рекомендация:\n{guides[game_number]['guides'][guide_id]}"

    await query.edit_message_text(
        text,
        parse_mode="HTML",
        reply_markup=InlineKeyboardMarkup([
            [InlineKeyboardButton("🧠 Пройти викторину", callback_data=f"квиз:{game_number}")],
            [InlineKeyboardButton("⬅️ Назад", callback_data=f"игра:{game_number}")]
        ])
    )

async def показать_тему(query, game_number):
    weak = find_weak_difficulty(game_number)
    topic = topic_analyzer(game_number)
    guide_id = guide_for_weak_difficulty(game_number)

    text = (
        "🧩 <b>СЛАБАЯ ТЕМА</b>\n\n"
        f"🎮 {games[game_number]['name']}\n"
        f"🎯 Сложность: {weak or 'пока нет данных'}\n"
        f"🧩 Тема: {topic or 'пока нет данных'}"
    )

    if guide_id:
        text += f"\n\n📚 Гайд:\n{guides[game_number]['guides'][guide_id]}"

    await query.edit_message_text(
        text,
        parse_mode="HTML",
        reply_markup=InlineKeyboardMarkup([
            [InlineKeyboardButton("🧠 Викторина", callback_data=f"квиз:{game_number}")],
            [InlineKeyboardButton("⬅️ Назад", callback_data=f"игра:{game_number}")]
        ])
    )

async def показать_прогресс(query):
    text = (
        f"📊 <b>ТВОЙ ПРОГРЕСС</b>\n\n"
        f"⭐ XP: <b>{progress['xp']}</b>\n"
        f"🏆 Уровень: <b>{progress['level']}</b>\n"
        f"🧠 Викторин: <b>{progress['quizzes_completed']}</b>\n"
        f"📚 Гайдов открыто: <b>{progress['guides_opened']}</b>\n"
        f"🤖 ML-записей: <b>{len(ml_history)}</b>\n\n"
    )

    for gid, game in games.items():
        stats = get_game_statistics(gid)
        text += f"🎮 <b>{game['name']}</b>\n"
        text += f"Последний результат: {stats['last_score']}%\n"
        text += f"Викторин: {stats['quizzes']}\n"

        for difficulty in DIFFICULTIES:
            total = stats[f"{difficulty.lower()}_total"]
            if total:
                text += f"{difficulty}: {difficulty_percent(stats, difficulty)}%\n"

        text += "\n"

    await query.edit_message_text(
        text,
        parse_mode="HTML",
        reply_markup=главное_меню()
    )

async def обработчик_кнопок(update: Update, context: ContextTypes.DEFAULT_TYPE):
    query = update.callback_query
    await query.answer()
    data = query.data

    if data == "меню":
        await query.edit_message_text(
            "🎮 <b>GAME HELPER MAX</b>\n\nВыбери действие:",
            parse_mode="HTML",
            reply_markup=главное_меню()
        )
        return

    if data == "игры":
        await показать_игры(query)
        return

    if data == "гайды":
        await показать_гайды(query)
        return

    if data == "викторина":
        await query.edit_message_text(
            "🧠 <b>ВИКТОРИНА</b>\n\nВыбери игру:",
            parse_mode="HTML",
            reply_markup=клавиатура_игр("квиз")
        )
        return

    if data == "ml":
        await query.edit_message_text(
            "🤖 <b>ML АНАЛИЗ</b>\n\nВыбери игру:",
            parse_mode="HTML",
            reply_markup=клавиатура_игр("мл_игры")
        )
        return

    if data == "тема":
        await query.edit_message_text(
            "🧩 <b>СЛАБАЯ ТЕМА</b>\n\nВыбери игру:",
            parse_mode="HTML",
            reply_markup=клавиатура_игр("тема_игры")
        )
        return

    if data == "прогресс":
        await показать_прогресс(query)
        return

    if data == "следующий_вопрос":
        await отправить_вопрос(query)
        return

    if data == "выйти_квиз":
        state = get_user_state(query.from_user.id)
        state["quiz_started"] = False
        state.pop("current_question", None)
        save_telegram_users()
        await query.edit_message_text(
            "❌ Викторина остановлена.",
            reply_markup=главное_меню()
        )
        return

    if data.startswith("ответ:"):
        await обработать_ответ(query, data.split(":", 1)[1])
        return

    parts = data.split(":")

    if len(parts) == 2:
        action, value = parts
        try:
            game_number = int(value)
        except ValueError:
            await query.answer("Ошибка.")
            return

        if action == "игра":
            await показать_игру(query, game_number)
        elif action == "квиз":
            await начать_викторину(query, game_number)
        elif action == "гайды_игры":
            await показать_список_гайдов(query, game_number)
        elif action == "мл_игры":
            await показать_ml(query, game_number)
        elif action == "тема_игры":
            await показать_тему(query, game_number)
        return

    if len(parts) == 3 and parts[0] == "гайд":
        try:
            game_number = int(parts[1])
            guide_id = int(parts[2])
        except ValueError:
            await query.answer("Ошибка.")
            return
        await показать_гайд(query, game_number, guide_id)

def запустить_бот():
    if not BOT_TOKEN:
        print("❌ BOT_TOKEN не найден.")
        print('PowerShell: $env:BOT_TOKEN="ТВОЙ_ТОКЕН"')
        return

    приложение = Application.builder().token(BOT_TOKEN).build()
    приложение.add_handler(CommandHandler("start", команда_start))
    приложение.add_handler(CallbackQueryHandler(обработчик_кнопок))

    print("🤖 GAME HELPER TELEGRAM MAX ЗАПУЩЕН!")
    print("📱 Отправь боту /start")
    print("🛑 Остановка: Ctrl+C")
    приложение.run_polling()

if __name__ == "__main__":
    запустить_бот()
